# Re-seeded validation: plots and metrics

Post-processing notebook for the re-seeded drogued drifter validation.
Reads simulation output produced by notebook 11a.

## Imports

In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

## Parameters

In [ ]:
BBOX = [9.8, 12.2, 54.1, 55.1]
LON_MIN = 9.0
LON_MAX = 13.0
LAT_MIN = 53.5
LAT_MAX = 56.0
LEAD_TIMES_HOURS = [1, 2, 4, 8, 16]
FORECAST_HOURS = 16
RESEED_INTERVAL_HOURS = 6
DT = 300.0
CMEMS_DIR = "data/cmems"

## Load simulation results

In [ ]:
df_sim = pd.read_csv(
    "output/reseeded_trajectories.csv",
    parse_dates=["time", "reseed_time"],
)
df_obs = pd.read_csv(
    "output/reseeded_obs.csv",
    parse_dates=["time"],
)

drifter_ids = sorted(df_sim["drifter"].unique())
sim_types = sorted(df_sim["sim_type"].unique())

# Reconstruct obs dict (drifter -> DataFrame indexed by time)
obs = {}
for d_num, grp in df_obs.groupby("drifter"):
    obs[d_num] = grp.set_index("time").rename(columns={"lon": "Longitude", "lat": "Latitude"})

# Reconstruct results list (one entry per re-seeding per sim_type)
results = []
for (d_num, sim_type, t_start), grp in df_sim.groupby(["drifter", "sim_type", "reseed_time"]):
    grp = grp.sort_values("time")
    obs_grp = obs[d_num]
    t_end = t_start + pd.Timedelta(hours=FORECAST_HOURS)
    results.append({
        "drifter": d_num,
        "sim_type": sim_type,
        "t_start": t_start,
        "sim_lon": grp["lon"].values,
        "sim_lat": grp["lat"].values,
        "sim_time": grp["time"].values,
        "obs_window": obs_grp.loc[t_start:t_end],
    })

for st in sim_types:
    n = sum(1 for r in results if r["sim_type"] == st)
    print(f"  {st}: {n} re-seedings")
print(f"Drifters: {drifter_ids}")

## CMEMS effective coastline

In [ ]:
ds_static = xr.open_dataset(CMEMS_DIR + "/cmems_mod_bal_phy_anfc_static.nc")
mask = ds_static.mask.sel(
    longitude=slice(LON_MIN, LON_MAX),
    latitude=slice(LAT_MIN, LAT_MAX),
).isel(depth=0).load()

land_cmap = mcolors.ListedColormap(["lightgray", "none"])

## Re-seeded trajectories per drifter

One figure per drifter showing the observed trajectory (red) and all
re-seeded simulations (blue), with CMEMS land mask and Natural Earth
coastline.

In [ ]:
coast_10m = cfeature.NaturalEarthFeature(
    "physical", "coastline", "10m",
    edgecolor="gray", facecolor="none",
)

SIM_STYLE = {
    "dd":      {"color": "blue",   "label": "Drogued drifter"},
    "surface": {"color": "green",  "label": "Surface pp"},
    "3m":      {"color": "orange", "label": "3m pp"},
}

for d_num in drifter_ids:
    fig, ax = plt.subplots(subplot_kw={"projection": ccrs.PlateCarree()})
    ax.add_feature(coast_10m)
    ax.pcolormesh(
        mask.longitude, mask.latitude, mask.values,
        cmap=land_cmap, vmin=0, vmax=1, alpha=0.4, shading="nearest",
        transform=ccrs.PlateCarree(),
    )
    ax.set_extent(BBOX)
    ax.gridlines(draw_labels=True)

    # Observed
    grp = obs[d_num]
    ax.plot(grp["Longitude"], grp["Latitude"], color="red", linewidth=0.7,
            label="Observed", transform=ccrs.PlateCarree(), zorder=2)

    # Simulated trajectories by type
    for sim_type, style in SIM_STYLE.items():
        d_results = [r for r in results
                     if r["drifter"] == d_num and r["sim_type"] == sim_type]
        for r in d_results:
            ax.plot(r["sim_lon"], r["sim_lat"], color=style["color"],
                    alpha=0.8, linewidth=0.4,
                    transform=ccrs.PlateCarree(), zorder=3)

    ax.plot(grp["Longitude"].iloc[0], grp["Latitude"].iloc[0], "ko",
            markersize=4, transform=ccrs.PlateCarree(), zorder=4)

    ax.set_title(f"D{d_num}")
    if d_num == drifter_ids[0]:
        # Legend entries
        for sim_type, style in SIM_STYLE.items():
            ax.plot([], [], color=style["color"], linewidth=1, label=style["label"])
        ax.legend(fontsize=7, loc="best")

    plt.tight_layout()
    plt.show()

In [ ]:
def haversine_km(lon1, lat1, lon2, lat2):
    """Great-circle distance in km between two points (arrays ok)."""
    R = 6371.0
    dlat = np.radians(lat2 - lat1)
    dlon = np.radians(lon2 - lon1)
    a = np.sin(dlat / 2) ** 2 + np.cos(np.radians(lat1)) * np.cos(np.radians(lat2)) * np.sin(dlon / 2) ** 2
    return R * 2 * np.arcsin(np.sqrt(a))


lead_hours = np.arange(0, FORECAST_HOURS + 1)

# Collect separation per sim_type and lead time
sep_by_lead = {st: {h: [] for h in lead_hours} for st in sim_types}
skill_scores = []

for r in results:
    obs_w = r["obs_window"]
    t0 = r["t_start"]
    st = r["sim_type"]

    sim_times_dt = pd.DatetimeIndex(r["sim_time"])
    sim_lon_s = pd.Series(r["sim_lon"], index=sim_times_dt).sort_index()
    sim_lat_s = pd.Series(r["sim_lat"], index=sim_times_dt).sort_index()
    sim_lon_s = sim_lon_s[~sim_lon_s.index.duplicated(keep="last")]
    sim_lat_s = sim_lat_s[~sim_lat_s.index.duplicated(keep="last")]

    cum_sep = 0.0
    running_path = 0.0
    cum_path_sum = 0.0
    n_valid = 0

    for h in lead_hours:
        t_target = t0 + pd.Timedelta(hours=h)

        if t_target not in obs_w.index:
            continue

        obs_lon = obs_w.loc[t_target, "Longitude"]
        obs_lat = obs_w.loc[t_target, "Latitude"]

        if t_target < sim_lon_s.index[0] or t_target > sim_lon_s.index[-1]:
            continue

        idx = sim_lon_s.index.get_indexer([t_target], method="nearest")[0]
        if idx < 0:
            continue
        nearest_t = sim_lon_s.index[idx]
        if abs((nearest_t - t_target).total_seconds()) > DT + 1:
            continue

        sim_lon = sim_lon_s.iloc[idx]
        sim_lat = sim_lat_s.iloc[idx]

        sep_km = haversine_km(sim_lon, sim_lat, obs_lon, obs_lat)
        sep_by_lead[st][h].append(sep_km)

        if h > 0:
            t_prev = t0 + pd.Timedelta(hours=h - 1)
            if t_prev in obs_w.index:
                obs_lon_prev = obs_w.loc[t_prev, "Longitude"]
                obs_lat_prev = obs_w.loc[t_prev, "Latitude"]
                running_path += haversine_km(obs_lon_prev, obs_lat_prev, obs_lon, obs_lat)

        cum_sep += sep_km
        cum_path_sum += running_path
        n_valid += 1

    if cum_path_sum > 0 and n_valid > 1:
        s = 1.0 - cum_sep / cum_path_sum
        skill_scores.append({
            "drifter": r["drifter"],
            "sim_type": st,
            "t_start": r["t_start"],
            "skill": s,
            "cum_sep_km": cum_sep,
            "cum_path_sum_km": cum_path_sum,
            "n_hours": n_valid,
        })

df_skill = pd.DataFrame(skill_scores)
for st in sim_types:
    n = len(df_skill[df_skill["sim_type"] == st])
    print(f"  {st}: {n} skill scores")

## Separation distance vs lead time

Mean and median separation distance across all re-seedings as a
function of forecast lead time. Shaded region shows the interquartile
range.

In [ ]:
fig, ax = plt.subplots()

for st, style in [("dd", "-"), ("surface", ":"), ("3m", "-.")]:
    mean_sep = np.array([
        np.mean(sep_by_lead[st][h]) if sep_by_lead[st][h] else np.nan
        for h in lead_hours
    ])
    valid = np.isfinite(mean_sep)
    ax.plot(lead_hours[valid], mean_sep[valid], style, label=st)

ax.set_xlabel("Lead time [hours]")
ax.set_ylabel("Mean separation distance [km]")
ax.legend()
plt.show()

## Separation distance vs lead time, per drifter

In [ ]:
sep_by_lead_drifter = {d: {h: [] for h in lead_hours} for d in drifter_ids}

for r in results:
    if r["sim_type"] != "dd":
        continue
    obs_w = r["obs_window"]
    t0 = r["t_start"]
    d_num = r["drifter"]

    sim_times_dt = pd.DatetimeIndex(r["sim_time"])
    sim_lon_s = pd.Series(r["sim_lon"], index=sim_times_dt).sort_index()
    sim_lat_s = pd.Series(r["sim_lat"], index=sim_times_dt).sort_index()
    sim_lon_s = sim_lon_s[~sim_lon_s.index.duplicated(keep="last")]
    sim_lat_s = sim_lat_s[~sim_lat_s.index.duplicated(keep="last")]

    for h in lead_hours:
        t_target = t0 + pd.Timedelta(hours=h)
        if t_target not in obs_w.index:
            continue
        obs_lon = obs_w.loc[t_target, "Longitude"]
        obs_lat = obs_w.loc[t_target, "Latitude"]
        if t_target < sim_lon_s.index[0] or t_target > sim_lon_s.index[-1]:
            continue
        idx = sim_lon_s.index.get_indexer([t_target], method="nearest")[0]
        if idx < 0:
            continue
        nearest_t = sim_lon_s.index[idx]
        if abs((nearest_t - t_target).total_seconds()) > DT + 1:
            continue
        sep_km = haversine_km(sim_lon_s.iloc[idx], sim_lat_s.iloc[idx], obs_lon, obs_lat)
        sep_by_lead_drifter[d_num][h].append(sep_km)

fig, ax = plt.subplots()
for d_num in drifter_ids:
    mean_d = np.array([
        np.mean(sep_by_lead_drifter[d_num][h]) if sep_by_lead_drifter[d_num][h] else np.nan
        for h in lead_hours
    ])
    v = np.isfinite(mean_d)
    ax.plot(lead_hours[v], mean_d[v], label=f"D{d_num}")

ax.set_xlabel("Lead time [hours]")
ax.set_ylabel("Mean separation distance [km]")
ax.set_title("Drogued drifter: per-drifter separation")
ax.legend()
plt.show()

## Liu & Weisberg skill score summary

Distribution of skill scores across all re-seedings, and per-drifter
summary statistics.

In [ ]:
fig, axes = plt.subplots(1, len(sim_types), sharey=True)

for ax, st in zip(axes, sim_types):
    data = df_skill[df_skill["sim_type"] == st]["skill"]
    ax.hist(data, bins=20, edgecolor="k", alpha=0.7)
    ax.axvline(data.median(), color="r", ls="--",
               label=f"median = {data.median():.2f}")
    ax.set_xlabel("Skill score")
    ax.set_title(st)
    ax.legend(fontsize=7)

axes[0].set_ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
summary = df_skill.groupby("sim_type")["skill"].agg(["mean", "median", "std", "count"]).round(3)
summary

## Separation at key lead times

Summary statistics for separation distance at each lead time.

In [ ]:
rows = []
for st in sim_types:
    for h in LEAD_TIMES_HOURS:
        vals = sep_by_lead[st].get(h, [])
        if vals:
            rows.append({
                "Type": st,
                "Lead [h]": h,
                "N": len(vals),
                "Mean [km]": np.mean(vals),
                "Median [km]": np.median(vals),
            })

df_sep_summary = pd.DataFrame(rows).round(2)
df_sep_summary